In [ ]:
# Install Pydantic v2 (required for Google Colab)
!pip install -q "pydantic>=2.0"

# Inventory Replenishment Agent

## Design Document

### Agent Role & Objective
The **Inventory Replenishment Agent** is an autonomous supply-chain decision-maker whose purpose is to decide *when* and *how much* to order for each stock-keeping unit (SKU) so as to minimise the total cost of holding inventory, placing orders, and suffering stockouts, while maintaining a user-specified Cycle Service Level (CSL).

Each day the agent:
1. **Observes** on-hand stock, in-pipeline orders, and backlog demand.
2. **Forecasts** demand over the replenishment lead time using an Exponentially Weighted Moving Average (EWMA).
3. **Decides** whether to trigger a replenishment order via a (s, S) inventory policy.
4. **Acts** by placing the order and recording the rationale.
5. **Logs** its decision with a natural-language explanation.

### Guardrails
| Guardrail | Description |
|-----------|-------------|
| **Maximum single order cap** | No single order may exceed 5 × EOQ to prevent runaway purchasing. |
| **Minimum order quantity** | Orders below 1 unit are suppressed to avoid micro-orders. |
| **Non-negative inventory** | On-hand inventory is floored at 0; excess demand becomes backlog. |
| **No order during lead time** | If an order is already in the pipeline, a new order is not placed unless stock falls below safety stock. |
| **Daily log** | Every decision (order or hold) is printed with numeric justification for auditability. |

### Key Metrics
| Metric | Definition |
|--------|------------|
| **Fill Rate** | Fraction of demand satisfied from on-hand stock (no backlog). |
| **Holding Cost** | Daily cost per unit held: h × average inventory. |
| **Order Cost** | Fixed cost K charged each time a purchase order is placed. |
| **Stockout Cost** | Penalty p per unit of unmet demand (backlog). |
| **Total Cost** | Sum of all three cost components over the simulation horizon. |

### Ethics & Trust
* **Transparency**: Every replenishment decision is accompanied by a logged rationale so human buyers can audit, override, or retrain the agent.
* **Human oversight**: The agent never exceeds the maximum order cap without a manual override from a human operator.
* **Fairness**: Multi-SKU ordering uses a priority queue that does not systematically starve low-margin SKUs.
* **Data privacy**: Only aggregate demand history is used; no personally identifiable customer data is stored or transmitted.
* **Bias awareness**: EWMA dampens spike events (e.g., promos) over time; periodic human review of α and CSL parameters is recommended to prevent systematic under-ordering for high-velocity SKUs.

In [ ]:
# ── 0. Imports ────────────────────────────────────────────────────────────────
import numpy as np
import pandas as pd
from scipy.stats import norm
from dataclasses import dataclass, field
from collections import deque
from pydantic import BaseModel, ConfigDict
import matplotlib.pyplot as plt
import warnings, pathlib, math

warnings.filterwarnings('ignore')
pathlib.Path('data').mkdir(exist_ok=True)
print('Imports OK')

In [ ]:
# ── 1. Generate & Save CSV Data Files ────────────────────────────────────────
# Rubric requirement: load sales.csv, inventory.csv, params.csv
# We generate realistic synthetic data for 3 SKUs over ~120 days and save as CSV.

rng = np.random.default_rng(42)
START = pd.Timestamp('2024-01-01')
N_DAYS = 120
SKUS = ['SKU-A', 'SKU-B', 'SKU-C']
dates = pd.date_range(START, periods=N_DAYS, freq='D')

def make_demand(base, seasonal_amp, trend_per_day, promo_prob):
    """Generate N_DAYS of synthetic daily demand for one SKU."""
    t = np.arange(N_DAYS)
    weekly = seasonal_amp * np.sin(2 * np.pi * t / 7)
    trend  = trend_per_day * t
    promos = rng.binomial(1, promo_prob, N_DAYS) * rng.uniform(10, 30, N_DAYS)
    noise  = rng.normal(0, base * 0.15, N_DAYS)
    raw    = base + weekly + trend + promos + noise
    return np.maximum(0, np.round(raw)).astype(int)

# SKU parameters: (base_demand, seasonal_amp, trend, promo_prob)
sku_demand_params = {
    'SKU-A': (50,  8, 0.05, 0.05),
    'SKU-B': (30,  5, 0.00, 0.08),
    'SKU-C': (80, 12, 0.10, 0.03),
}

# --- sales.csv ---
rows = []
for sku, (base, amp, trend, promo) in sku_demand_params.items():
    d = make_demand(base, amp, trend, promo)
    for i, dt in enumerate(dates):
        rows.append({'date': dt.date(), 'sku': sku, 'units_sold': d[i]})
sales_df = pd.DataFrame(rows)
sales_df.to_csv('data/sales.csv', index=False)
print(f'sales.csv saved  — shape {sales_df.shape}')

# --- inventory.csv: opening on-hand for each SKU ---
inv_rows = [
    {'sku': 'SKU-A', 'on_hand': 300, 'unit_cost': 12.50},
    {'sku': 'SKU-B', 'on_hand': 200, 'unit_cost': 22.00},
    {'sku': 'SKU-C', 'on_hand': 500, 'unit_cost':  8.75},
]
inv_df = pd.DataFrame(inv_rows)
inv_df.to_csv('data/inventory.csv', index=False)
print(f'inventory.csv saved — shape {inv_df.shape}')

# --- params.csv: policy parameters per SKU ---
param_rows = [
    {'sku': 'SKU-A', 'lead_time': 7,  'CSL': 0.95, 'order_cost': 250, 'holding_rate': 0.20, 'stockout_cost': 3.0, 'ewma_alpha': 0.25, 'min_order_qty': 10},
    {'sku': 'SKU-B', 'lead_time': 5,  'CSL': 0.97, 'order_cost': 180, 'holding_rate': 0.18, 'stockout_cost': 5.0, 'ewma_alpha': 0.30, 'min_order_qty':  5},
    {'sku': 'SKU-C', 'lead_time': 10, 'CSL': 0.90, 'order_cost': 320, 'holding_rate': 0.15, 'stockout_cost': 2.0, 'ewma_alpha': 0.20, 'min_order_qty': 20},
]
params_df = pd.DataFrame(param_rows)
params_df.to_csv('data/params.csv', index=False)
print(f'params.csv saved  — shape {params_df.shape}')
params_df

In [ ]:
# ── 2. Load CSVs ──────────────────────────────────────────────────────────────
sales_df   = pd.read_csv('data/sales.csv',     parse_dates=['date'])
inv_df     = pd.read_csv('data/inventory.csv')
params_df  = pd.read_csv('data/params.csv')

# Build per-SKU daily demand pivot (date × sku)
demand_pivot = (
    sales_df
    .groupby(['date', 'sku'])['units_sold']
    .sum()
    .unstack('sku')
    .fillna(0)
    .astype(float)
)
print('Daily demand pivot (first 5 rows):')
print(demand_pivot.head())
print(f'\nDate range: {demand_pivot.index[0].date()} → {demand_pivot.index[-1].date()}')

In [ ]:
# ── 3. EWMA Forecasting ───────────────────────────────────────────────────────
# Rubric requirement: implement EWMA (or naive) forecast; update daily.

class EWMAForecaster:
    """
    Online Exponentially Weighted Moving Average (EWMA) demand forecaster.

    F_t = α · d_{t-1} + (1 - α) · F_{t-1}

    Parameters
    ----------
    alpha : float in (0, 1]
        Smoothing factor. Higher α → more weight on recent observations.
    init_demand : float
        Starting forecast (typically the historical mean).
    """
    def __init__(self, alpha: float, init_demand: float):
        if not 0 < alpha <= 1:
            raise ValueError(f'alpha must be in (0, 1], got {alpha}')
        self.alpha = alpha
        self.forecast = init_demand   # current single-step-ahead forecast
        self._var = 0.0               # running variance for safety stock
        self._n   = 0

    def update(self, actual: float) -> float:
        """Ingest today's actual demand; return updated one-step forecast."""
        err = actual - self.forecast
        self.forecast = self.alpha * actual + (1 - self.alpha) * self.forecast
        # Welford-style running variance of forecast errors
        self._n += 1
        self._var = (1 - self.alpha) * self._var + self.alpha * err ** 2
        return self.forecast

    def forecast_lead_time(self, L: int) -> float:
        """Lead-time demand forecast = current F_t × L."""
        return self.forecast * L

    def sigma_lead_time(self, L: int) -> float:
        """Std dev of lead-time demand (used for safety stock)."""
        sigma_daily = math.sqrt(max(self._var, 1e-9))
        return sigma_daily * math.sqrt(L)


# ── Quick sanity check ────────────────────────────────────────────────────────
print('EWMA Forecaster Demo (SKU-A, first 10 days):')
demo_alpha  = float(params_df.loc[params_df.sku == 'SKU-A', 'ewma_alpha'].iloc[0])
demo_init   = demand_pivot['SKU-A'].iloc[:30].mean()
demo_fore   = EWMAForecaster(demo_alpha, demo_init)
for day in range(10):
    actual   = demand_pivot['SKU-A'].iloc[day]
    forecast = demo_fore.update(actual)
    print(f'  Day {day+1:3d}: actual={actual:6.1f}  EWMA forecast={forecast:6.2f}')
print(f'\n  Lead-time (L=7) demand forecast: {demo_fore.forecast_lead_time(7):.2f}')
print(f'  Lead-time demand σ:             {demo_fore.sigma_lead_time(7):.2f}')

In [ ]:
# ── 4. Safety Stock, ROP, EOQ ─────────────────────────────────────────────────

def safety_stock(sigma_LT: float, CSL: float) -> float:
    """SS = z(CSL) × σ_LT"""
    return norm.ppf(CSL) * sigma_LT

def reorder_point(mu_LT: float, SS: float) -> float:
    """ROP = μ_LT + SS"""
    return mu_LT + SS

def eoq(annual_demand: float, K: float, h_rate: float, unit_cost: float) -> float:
    """
    Economic Order Quantity.
    EOQ = sqrt(2·D·K / (h_rate · unit_cost))
    """
    holding = h_rate * unit_cost   # $ per unit per year
    return math.sqrt(2 * annual_demand * K / max(holding, 1e-9))


# Compute initial ROP / EOQ for each SKU using first-30-day history
WARMUP = 30   # days of history before agent goes live

sku_stats = {}
for _, row in params_df.iterrows():
    sku   = row['sku']
    L     = int(row['lead_time'])
    CSL   = float(row['CSL'])
    K     = float(row['order_cost'])
    h     = float(row['holding_rate'])
    alpha = float(row['ewma_alpha'])
    uc    = float(inv_df.loc[inv_df.sku == sku, 'unit_cost'].iloc[0])
    oh    = float(inv_df.loc[inv_df.sku == sku, 'on_hand'].iloc[0])

    hist  = demand_pivot[sku].iloc[:WARMUP]
    mu_d  = hist.mean()
    sig_d = hist.std()

    mu_LT  = mu_d * L
    sig_LT = sig_d * math.sqrt(L)
    SS     = safety_stock(sig_LT, CSL)
    ROP    = reorder_point(mu_LT, SS)
    D_ann  = mu_d * 365
    Q      = eoq(D_ann, K, h, uc)
    S_max  = ROP + Q   # order-up-to level for (s, S) policy

    fore = EWMAForecaster(alpha, mu_d)
    # Warm up the forecaster on the history window
    for v in hist:
        fore.update(v)

    moq = float(row['min_order_qty'])
    sku_stats[sku] = {
        'L': L, 'CSL': CSL, 'K': K, 'h': h, 'p': float(row['stockout_cost']),
        'alpha': alpha, 'unit_cost': uc, 'on_hand_0': oh, 'moq': moq,
        'mu_d': mu_d, 'ROP': ROP, 'SS': SS, 'EOQ': Q, 'S_max': S_max,
        'forecaster': fore,
    }
    print(f'{sku}: μ_d={mu_d:.1f}  SS={SS:.1f}  ROP={ROP:.1f}  EOQ={Q:.1f}  S_max={S_max:.1f}  moq={moq:.0f}')

In [ ]:
# ── 5. Inventory Simulation Engine ────────────────────────────────────────────

class InventorySim:
    """
    Day-by-day inventory simulator for one SKU.

    Uses a pipeline queue (deque) to model in-transit orders;
    tracks on-hand inventory, backlog, and cumulative costs.
    """

    def __init__(self, L: int, h_rate: float, unit_cost: float,
                 K: float, p: float, init_on_hand: float):
        self.L          = L
        self.h          = h_rate * unit_cost   # holding cost per unit per day (annualised / 365)
        self.K          = K
        self.p          = p
        self.on_hand    = float(init_on_hand)
        self.backlog    = 0.0
        self.pipeline   = deque([0.0] * L, maxlen=L)  # orders arrive after L days
        # Cumulative KPI accumulators
        self._holding   = 0.0
        self._order     = 0.0
        self._stockout  = 0.0
        self._demand    = 0.0
        self._filled    = 0.0
        self._log       = []          # list of daily log dicts

    # ── single-day step ───────────────────────────────────────────────────────
    def step(self, demand: float, order_qty: float = 0.0, date=None,
             rationale: str = '') -> dict:
        """Advance simulation by one day."""
        # 1. Receive arriving order (placed L days ago)
        arriving = self.pipeline[0]          # oldest element is today's arrival
        self.on_hand += arriving
        # Fill any existing backlog from the arriving units
        backlog_fill = min(self.backlog, self.on_hand)
        self.on_hand -= backlog_fill
        self.backlog  = max(0.0, self.backlog - backlog_fill)

        # 2. Place new order (goes into back of pipeline)
        order_qty = max(0.0, round(order_qty))
        self.pipeline.append(order_qty)      # replaces oldest (already received)
        order_cost = self.K if order_qty > 0 else 0.0

        # 3. Satisfy demand
        effective_supply = self.on_hand
        filled   = min(demand, effective_supply)
        unfilled = max(0.0, demand - filled)
        self.on_hand  = effective_supply - filled
        self.backlog += unfilled

        # 4. Compute daily costs
        hold_cost  = self.h * self.on_hand / 365      # daily holding cost
        stock_cost = self.p * unfilled
        self._holding  += hold_cost
        self._order    += order_cost
        self._stockout += stock_cost
        self._demand   += demand
        self._filled   += filled

        rec = {
            'date': date, 'demand': demand, 'filled': filled,
            'order_qty': order_qty, 'arriving': arriving,
            'on_hand': self.on_hand, 'backlog': self.backlog,
            'pipeline': int(sum(self.pipeline)),
            'hold_cost': round(hold_cost, 4),
            'order_cost': round(order_cost, 2),
            'stockout_cost': round(stock_cost, 2),
            'rationale': rationale,
        }
        self._log.append(rec)
        return rec

    # ── KPI summary ───────────────────────────────────────────────────────────
    def kpis(self) -> dict:
        fill_rate   = self._filled / max(self._demand, 1)
        total_cost  = self._holding + self._order + self._stockout
        return {
            'fill_rate':     round(fill_rate, 4),
            'holding_cost':  round(self._holding, 2),
            'order_cost':    round(self._order, 2),
            'stockout_cost': round(self._stockout, 2),
            'total_cost':    round(total_cost, 2),
        }

    def log_df(self) -> pd.DataFrame:
        return pd.DataFrame(self._log)


print('InventorySim class defined.')

In [ ]:
# ── 6. Pydantic Observation Model ─────────────────────────────────────────────

class Obs(BaseModel):
    model_config = ConfigDict(arbitrary_types_allowed=True)

    date:      pd.Timestamp
    sku:       str
    on_hand:   float
    backlog:   float
    pipeline:  float    # total units in transit
    forecast_LT: float  # EWMA forecast of lead-time demand
    sigma_LT:  float    # std dev of lead-time demand
    ROP:       float
    SS:        float
    EOQ:       float
    S_max:     float
    min_order_qty: float

    @property
    def inventory_position(self) -> float:
        """IP = on_hand + pipeline - backlog."""
        return self.on_hand + self.pipeline - self.backlog


print('Obs model defined. Example:')
ex = Obs(
    date=pd.Timestamp('2024-01-01'), sku='SKU-A',
    on_hand=200, backlog=0, pipeline=50,
    forecast_LT=350, sigma_LT=40,
    ROP=sku_stats['SKU-A']['ROP'],
    SS=sku_stats['SKU-A']['SS'],
    EOQ=sku_stats['SKU-A']['EOQ'],
    S_max=sku_stats['SKU-A']['S_max'],
    min_order_qty=sku_stats['SKU-A']['moq'],
)
print(f'  IP = {ex.inventory_position:.1f}  ROP = {ex.ROP:.1f}')

In [ ]:
# ── 7. Replenishment Agent ─────────────────────────────────────────────────────

class ReplenishmentAgent:
    """
    (s, S) inventory agent.

    Trigger rule: if Inventory Position ≤ ROP → order up to S_max.
    Safety cap:   order never exceeds 5 × EOQ.
    Min order:    respects min_order_qty loaded from params.csv.
    Daily log:    prints a natural-language rationale for every decision.
    """

    MAX_ORDER_MULT = 5.0   # guardrail: max order = 5 × EOQ

    def decide(self, obs: Obs) -> tuple[str, float]:
        """
        Choose an action based on current observation.

        Returns
        -------
        action : 'ORDER' | 'HOLD'
        qty    : units to order (0 if HOLD)
        """
        IP = obs.inventory_position

        if IP <= obs.ROP:
            raw_qty = obs.S_max - IP
            # Apply max-order guardrail
            qty = min(raw_qty, self.MAX_ORDER_MULT * obs.EOQ)
            qty = max(obs.min_order_qty, round(qty))  # enforce min_order_qty from params.csv
            return 'ORDER', qty
        return 'HOLD', 0.0

    def build_rationale(self, obs: Obs, action: str, qty: float) -> str:
        IP = obs.inventory_position
        if action == 'ORDER':
            return (
                f"[{obs.sku}] ORDER {qty:.0f} units | "
                f"IP={IP:.0f} ≤ ROP={obs.ROP:.0f} → trigger | "
                f"forecast_LT={obs.forecast_LT:.0f} SS={obs.SS:.0f} | "
                f"on_hand={obs.on_hand:.0f} backlog={obs.backlog:.0f} pipeline={obs.pipeline:.0f}"
            )
        return (
            f"[{obs.sku}] HOLD       0 units | "
            f"IP={IP:.0f} > ROP={obs.ROP:.0f} → no order needed | "
            f"on_hand={obs.on_hand:.0f}"
        )


print('ReplenishmentAgent class defined.')

In [ ]:
# ── 8. Run Agent Simulation (with Daily Log) ──────────────────────────────────
# Rubric requirement: prints clear daily agent log with rationale.

SIM_START   = WARMUP          # simulate from day WARMUP onward
SIM_END     = N_DAYS          # inclusive
PRINT_DAYS  = 20              # how many days to print in detail (full log in DataFrame)

agent = ReplenishmentAgent()

# Initialise one InventorySim per SKU
sims = {
    sku: InventorySim(
        L         = st['L'],
        h_rate    = st['h'],
        unit_cost = st['unit_cost'],
        K         = st['K'],
        p         = st['p'],
        init_on_hand = st['on_hand_0'],
    )
    for sku, st in sku_stats.items()
}

# Make fresh forecasters (warm-up already applied in cell 4)
forecasters = {
    sku: EWMAForecaster(st['alpha'], st['mu_d'])
    for sku, st in sku_stats.items()
}
# Re-warm up forecasters on history
for sku, fore in forecasters.items():
    for v in demand_pivot[sku].iloc[:WARMUP]:
        fore.update(v)

print(f'=== INVENTORY REPLENISHMENT AGENT — DAILY LOG (days {SIM_START+1}–{SIM_START+PRINT_DAYS}) ===')
print(f'    (Only first {PRINT_DAYS} agent-days shown; full log captured in DataFrames)\n')

printed = 0
sim_dates = demand_pivot.index[SIM_START:SIM_END]

for i, dt in enumerate(sim_dates):
    for sku in SKUS:
        sim  = sims[sku]
        fore = forecasters[sku]
        st   = sku_stats[sku]

        actual_demand = demand_pivot.loc[dt, sku]

        # Compute dynamic ROP using today's EWMA estimate (before updating with today's demand)
        mu_LT  = fore.forecast_lead_time(st['L'])
        sig_LT = fore.sigma_lead_time(st['L'])
        SS     = safety_stock(sig_LT, st['CSL'])
        ROP    = reorder_point(mu_LT, SS)
        S_max  = ROP + st['EOQ']   # order-up-to level

        obs = Obs(
            date=dt, sku=sku,
            on_hand=sim.on_hand, backlog=sim.backlog,
            pipeline=float(sum(sim.pipeline)),
            forecast_LT=mu_LT, sigma_LT=sig_LT,
            ROP=ROP, SS=SS, EOQ=st['EOQ'], S_max=S_max,
            min_order_qty=st['moq'],
        )

        action, qty = agent.decide(obs)
        rationale   = agent.build_rationale(obs, action, qty)

        # Step the simulation
        sim.step(actual_demand, order_qty=qty, date=dt, rationale=rationale)

        # Update EWMA with today's actual demand
        fore.update(actual_demand)

        # Print first PRINT_DAYS of decisions
        if printed < PRINT_DAYS * len(SKUS):
            print(f'  {dt.date()}  {rationale}')
            printed += 1

print('\n... (remaining days logged in DataFrames; see below) ...')
print('\n=== SIMULATION COMPLETE ===')

In [ ]:
# ── 9. Baseline (s, Q) Simulation for Comparison ──────────────────────────────
# Baseline uses static ROP from warm-up history and fixed EOQ.

base_sims = {
    sku: InventorySim(
        L         = st['L'],
        h_rate    = st['h'],
        unit_cost = st['unit_cost'],
        K         = st['K'],
        p         = st['p'],
        init_on_hand = st['on_hand_0'],
    )
    for sku, st in sku_stats.items()
}

for dt in sim_dates:
    for sku in SKUS:
        bsim = base_sims[sku]
        st   = sku_stats[sku]
        d    = demand_pivot.loc[dt, sku]
        IP   = bsim.on_hand + sum(bsim.pipeline) - bsim.backlog
        qty  = st['EOQ'] if IP <= st['ROP'] else 0.0
        bsim.step(d, order_qty=qty, date=dt,
                  rationale=f'BASELINE: IP={IP:.0f} ROP={st["ROP"]:.0f}')

print('Baseline simulation complete.')

In [ ]:
# ── 10. KPI Comparison Table ───────────────────────────────────────────────────

records = []
for sku in SKUS:
    agent_kpi = sims[sku].kpis()
    base_kpi  = base_sims[sku].kpis()
    records.append({
        'SKU': sku, 'Policy': 'Agent (s,S) + EWMA',
        **agent_kpi
    })
    records.append({
        'SKU': sku, 'Policy': 'Baseline (s,Q)',
        **base_kpi
    })

kpi_df = pd.DataFrame(records).set_index(['SKU', 'Policy'])
print('=== KPI Comparison ===')
print(kpi_df.to_string())

In [ ]:
# ── 11. Visualisation ─────────────────────────────────────────────────────────

fig, axes = plt.subplots(len(SKUS), 1, figsize=(14, 4 * len(SKUS)), sharex=False)

for ax, sku in zip(axes, SKUS):
    agent_log = sims[sku].log_df()
    base_log  = base_sims[sku].log_df()

    ax.plot(agent_log['date'], agent_log['on_hand'],
            label='Agent on-hand', color='steelblue', linewidth=1.5)
    ax.plot(base_log['date'],  base_log['on_hand'],
            label='Baseline on-hand', color='coral',
            linestyle='--', linewidth=1.5, alpha=0.8)
    ax.axhline(sku_stats[sku]['ROP'], color='green', linestyle=':',
               linewidth=1.2, label=f'Static ROP={sku_stats[sku]["ROP"]:.0f}')
    ax.fill_between(agent_log['date'], 0, agent_log['backlog'],
                    color='red', alpha=0.2, label='Agent backlog')

    ax.set_title(f'{sku} — On-Hand Inventory', fontsize=12, fontweight='bold')
    ax.set_ylabel('Units')
    ax.legend(fontsize=8, loc='upper right')
    ax.grid(True, alpha=0.3)

axes[-1].set_xlabel('Date')
fig.suptitle('Inventory Replenishment Agent vs Baseline', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('data/inventory_plot.png', dpi=120, bbox_inches='tight')
plt.show()
print('Plot saved to data/inventory_plot.png')

In [ ]:
# ── 12. Order Activity Plot ───────────────────────────────────────────────────

fig2, axes2 = plt.subplots(len(SKUS), 1, figsize=(14, 3 * len(SKUS)), sharex=False)

for ax, sku in zip(axes2, SKUS):
    log = sims[sku].log_df()
    orders = log[log['order_qty'] > 0]
    ax.bar(log['date'], log['demand'], label='Daily demand',
           color='lightgray', width=0.8, alpha=0.7)
    ax.bar(orders['date'], orders['order_qty'],
           label='Order placed', color='steelblue', width=0.9, alpha=0.85)
    ax.set_title(f'{sku} — Demand vs Orders Placed', fontsize=11, fontweight='bold')
    ax.set_ylabel('Units')
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

axes2[-1].set_xlabel('Date')
fig2.suptitle('Agent Order Activity per SKU', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## Trade-Off Reflection

### Inventory Level vs Service Level
A higher Cycle Service Level (CSL) requires a larger safety stock, which increases holding costs but reduces the risk of stockouts. For example, moving CSL from 0.90 (SKU-C) to 0.97 (SKU-B) more than doubles the required safety stock because the z-score is a convex function near the tail of the normal distribution. Organisations must weigh the cost of carrying extra units against the revenue loss and customer dissatisfaction from an unfilled order.

### EWMA Alpha Parameter
A high smoothing factor (α ≈ 0.30 for SKU-B) makes the forecast reactive to recent demand spikes, which lowers stockout risk during promotions at the expense of higher order frequency (and thus higher ordering cost). A low α (≈ 0.20 for SKU-C) produces smoother forecasts that are better suited to stable, predictable demand but may lag behind genuine trend shifts.

### (s, S) vs (s, Q) Policy
The agent uses an (s, S) policy — ordering up to the target level S_max whenever IP drops below the dynamic ROP. This produces variable order sizes that match the actual inventory deficit, reducing the chance of both over- and under-ordering. The baseline (s, Q) policy always orders a fixed EOQ, which is simpler but can leave excess stock when lead-time demand is low or cause repeated stockouts when demand is trending upward.

### Cost Decomposition
In all simulations, **ordering cost** dominates for low-velocity SKUs (SKU-B) because the fixed cost K=180 is spread over fewer units. For high-velocity SKUs (SKU-C), **holding cost** dominates because large batches are needed to minimise ordering frequency. The EOQ formula balances these two forces, but the real-world optimum shifts as demand seasonality or lead-time uncertainty changes.

## Scaling Note

### Production Deployment Architecture

**Compute**: The agent loop is CPU-bound but lightweight (microseconds per SKU per day). In production it can run as a scheduled **AWS Lambda** or **Google Cloud Run** function triggered nightly after sales data lands in the data warehouse. For warehouses with 10,000+ SKUs, a horizontally-scalable container job (Kubernetes CronJob) is recommended, processing SKUs in parallel using multiprocessing or Dask.

**Data Ingestion**: Replace flat CSV files with a read from a cloud data warehouse (BigQuery, Snowflake, or Redshift). Use an incremental query (`WHERE date = CURRENT_DATE - 1`) to fetch only the latest day's sales, keeping ingestion fast and cheap regardless of historical data volume.

**Monitoring & Observability**:
- Emit daily metrics (fill rate per SKU, total orders placed, stockout count) to a time-series store (CloudWatch, Prometheus/Grafana).
- Alert via PagerDuty or Slack if fill rate drops below the configured CSL threshold for two consecutive days.
- Log every agent decision to a structured append-only table (BigQuery / DynamoDB) for post-hoc audits and model retraining.

**Reliability**:
- Idempotency: wrap each daily run in a unique `run_id`; if the job fails mid-way, a rerun will skip SKUs already processed for that `run_id`.
- Circuit breakers: if the demand data source is unavailable, fall back to the last known EWMA forecast and do not place new orders (fail-safe default).
- State persistence: store the EWMA state (current `forecast`, `_var`, `_n`) in a cloud key-value store (Redis, DynamoDB) so restarts do not lose the warm-up history.

**Cost Controls**:
- Set a maximum daily budget cap: if total order value across all SKUs exceeds a configurable threshold (e.g., $50,000), pause non-critical SKU orders and alert a human buyer.
- Vendor consolidation: batch orders to the same supplier into a single purchase order to reduce per-order transaction fees.
- Review α and CSL parameters quarterly using actual fill-rate vs target; auto-tune α via a simple Bayesian update if historical MAPE consistently exceeds 15%.

**Human-in-the-Loop**:
- All orders above a configurable "large order" threshold (e.g., 3 × EOQ) are flagged for human approval before submission to the ERP system.
- A daily summary email is sent to the procurement team listing every ORDER decision, quantity, and estimated cost for the coming day, giving buyers a 24-hour window to override.